# Star Cluster Injection Pipeline - RSP Test Notebook

This notebook tests the injection pipeline on the Rubin Science Platform using **actual Rubin PSFs** from coadd images.

## Setup Instructions

**Before running this notebook, you need to get the pipeline code onto RSP:**

### Option A: Clone from GitHub (Recommended)
Run this in a terminal on RSP:
```bash
cd ~
git clone https://github.com/YOUR_USERNAME/star-cluster-injection-pipeline.git
```

### Option B: Upload manually
1. Zip your local `star-cluster-injection-pipeline` folder
2. Upload to RSP using the JupyterLab file browser
3. Unzip: `unzip star-cluster-injection-pipeline.zip`

### Option C: Install as a package
If you have a `setup.py`, run:
```bash
cd ~/star-cluster-injection-pipeline
pip install -e .
```

## 1. Setup and Imports

In [ ]:
# First, let's set up the pipeline path
import os
import sys

# Get your RSP username
USERNAME = os.environ.get('USER', 'your_username')
print(f"RSP Username: {USERNAME}")

# Set the pipeline path - UPDATE IF NEEDED
PIPELINE_PATH = f'/home/{USERNAME}/star-cluster-injection-pipeline'

# Check if the pipeline exists
if os.path.exists(PIPELINE_PATH):
    print(f"Pipeline found at: {PIPELINE_PATH}")
    sys.path.insert(0, PIPELINE_PATH)
else:
    print(f"ERROR: Pipeline not found at {PIPELINE_PATH}")
    print("\nPlease either:")
    print("  1. Clone from GitHub: git clone https://github.com/YOUR_USERNAME/star-cluster-injection-pipeline.git")
    print("  2. Upload the folder manually via JupyterLab")
    print("\nThen re-run this cell.")

# Standard imports
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# LSST imports
from lsst.daf.butler import Butler
from lsst.geom import Point2D

# Pipeline imports - these will fail if pipeline not found
try:
    from src.light_profiles import PlummerProfile, KingProfile, mag_to_flux
    from src.inject import inject_cluster, create_injection_catalog, inject_from_catalog
    from src.psf_convolution import get_psf_from_coadd, convolve_with_coadd_psf, get_psf_fwhm_from_coadd
    print("All pipeline imports successful!")
except ImportError as e:
    print(f"Import error: {e}")
    print(f"\nMake sure the pipeline is at: {PIPELINE_PATH}")

## 2. Load a Coadd Image

Update the `DATA_ID` below to point to a valid coadd in your available collections.

In [ ]:
# Configure Butler - UPDATE THESE FOR YOUR DATA
# For DP0.2:
REPO = 'dp02'
COLLECTION = '2.2i/runs/DP0.2'

# Data ID for a coadd - UPDATE THESE
DATA_ID = {
    'tract': 4431,
    'patch': 17,
    'band': 'i'
}

print(f"Loading Butler from {REPO}...")
butler = Butler(REPO, collections=COLLECTION)

print(f"Loading coadd: {DATA_ID}")
exposure = butler.get('deepCoadd', dataId=DATA_ID)

# Get image array
image = exposure.image.array.copy()
print(f"Image shape: {image.shape}")
print(f"Image dtype: {image.dtype}")
print(f"Image range: [{image.min():.2f}, {image.max():.2f}]")

In [ ]:
# Display the coadd
fig, ax = plt.subplots(figsize=(10, 10))
vmin, vmax = np.percentile(image, [1, 99])
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title(f'Coadd: tract={DATA_ID["tract"]}, patch={DATA_ID["patch"]}, band={DATA_ID["band"]}')
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
plt.tight_layout()
plt.show()

## 3. Test PSF Extraction from Coadd

This is the key test - verifying we can extract the **actual PSF** from the coadd.

In [ ]:
# Test PSF extraction at different positions
positions = [
    (image.shape[0]//4, image.shape[1]//4),
    (image.shape[0]//2, image.shape[1]//2),
    (3*image.shape[0]//4, 3*image.shape[1]//4),
]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

for i, (y, x) in enumerate(positions):
    pos = Point2D(x, y)
    
    # Extract PSF
    psf_array = get_psf_from_coadd(exposure, pos)
    fwhm = get_psf_fwhm_from_coadd(exposure, pos)
    
    print(f"Position ({x}, {y}): PSF shape={psf_array.shape}, FWHM={fwhm:.2f} pixels, sum={np.sum(psf_array):.4f}")
    
    # Plot PSF image
    ax = axes[0, i]
    im = ax.imshow(psf_array, cmap='hot', origin='lower')
    ax.set_title(f'PSF at ({x}, {y})\nFWHM={fwhm:.2f} px')
    plt.colorbar(im, ax=ax, shrink=0.8)
    
    # Plot radial profile
    ax = axes[1, i]
    cy, cx = psf_array.shape[0]//2, psf_array.shape[1]//2
    r = np.arange(0, min(cx, cy))
    radial = psf_array[cy, cx:cx+len(r)]
    ax.semilogy(r, radial / radial.max(), 'b-', lw=2)
    ax.axvline(fwhm/2, color='r', linestyle='--', label=f'HWHM={fwhm/2:.1f}')
    ax.set_xlabel('Radius (pixels)')
    ax.set_ylabel('Normalized Intensity')
    ax.set_title('PSF Radial Profile')
    ax.legend()

plt.suptitle('TEST: PSF Extraction from Coadd at Different Positions', fontsize=14)
plt.tight_layout()
plt.show()

## 4. Test PSF Convolution with Actual Rubin PSF

In [ ]:
# Create test profiles and convolve with ACTUAL Rubin PSF
fig, axes = plt.subplots(3, 4, figsize=(18, 12))

r_halfs = [2, 5, 10, 20]  # Different cluster sizes
center_pos = Point2D(image.shape[1]//2, image.shape[0]//2)

for i, r_half in enumerate(r_halfs):
    profile = PlummerProfile(r_half=r_half, age=1.0, central_brightness=100)
    intrinsic = profile.generate_2d((101, 101))
    
    # Convolve with ACTUAL Rubin PSF from the coadd
    convolved = convolve_with_coadd_psf(intrinsic, exposure, center_pos)
    
    # Row 0: Intrinsic profile
    ax = axes[0, i]
    im = ax.imshow(intrinsic, cmap='hot', origin='lower', norm=LogNorm(vmin=0.01, vmax=intrinsic.max()))
    ax.set_title(f'Intrinsic\nr_half={r_half}px')
    plt.colorbar(im, ax=ax, shrink=0.7)
    
    # Row 1: Convolved with Rubin PSF
    ax = axes[1, i]
    im = ax.imshow(convolved, cmap='hot', origin='lower', norm=LogNorm(vmin=0.01, vmax=intrinsic.max()))
    peak_ratio = convolved.max() / intrinsic.max()
    flux_ratio = np.sum(convolved) / np.sum(intrinsic)
    ax.set_title(f'Convolved (Rubin PSF)\npeak={peak_ratio:.2f}, flux={flux_ratio:.3f}')
    plt.colorbar(im, ax=ax, shrink=0.7)
    
    # Row 2: Radial comparison
    ax = axes[2, i]
    center = 50
    r = np.arange(0, 40)
    ax.semilogy(r, intrinsic[center, center:center+len(r)], 'b-', lw=2, label='Intrinsic')
    ax.semilogy(r, convolved[center, center:center+len(r)], 'r--', lw=2, label='Convolved')
    ax.axvline(r_half, color='gray', linestyle=':', label=f'r_half={r_half}')
    ax.set_xlabel('Radius (pixels)')
    ax.set_ylabel('Surface Brightness')
    ax.legend(fontsize=8)
    ax.set_title('Radial Profile')

plt.suptitle('TEST: Convolution with ACTUAL Rubin PSF from Coadd', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Single Cluster Injection Test

In [ ]:
# Pick a location in the image (avoid edges)
inject_y = image.shape[0] // 2
inject_x = image.shape[1] // 2

# Create a cluster
profile = PlummerProfile(r_half=15, age=1.0, magnitude=20)
print(f"Cluster: r_half=15px, magnitude=20, total_flux={profile.total_flux:.1f}")

# Inject using ACTUAL Rubin PSF
injected_image, cluster_image = inject_cluster(
    image,
    position=(inject_y, inject_x),
    profile=profile,
    exposure=exposure,  # <-- This uses the actual Rubin PSF!
    add_noise=False
)

print(f"Injected cluster peak: {cluster_image.max():.2f}")
print(f"Injected cluster total flux: {np.sum(cluster_image):.2f}")

In [ ]:
# Visualize the injection
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

# Cutout region around injection
margin = 100
y_slice = slice(max(0, inject_y-margin), min(image.shape[0], inject_y+margin))
x_slice = slice(max(0, inject_x-margin), min(image.shape[1], inject_x+margin))

vmin, vmax = np.percentile(image[y_slice, x_slice], [1, 99])

# Original
ax = axes[0]
ax.imshow(image[y_slice, x_slice], cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title('Original Coadd (cutout)')

# Cluster model
ax = axes[1]
cluster_cutout = cluster_image[y_slice, x_slice]
im = ax.imshow(cluster_cutout, cmap='hot', origin='lower', 
               norm=LogNorm(vmin=0.01, vmax=max(cluster_cutout.max(), 0.1)))
ax.set_title(f'Cluster Model (PSF convolved)\npeak={cluster_cutout.max():.2f}')
plt.colorbar(im, ax=ax)

# Injected
ax = axes[2]
ax.imshow(injected_image[y_slice, x_slice], cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.scatter(margin, margin, s=200, facecolors='none', edgecolors='red', linewidth=2)
ax.set_title('Injected Image')

# Difference
ax = axes[3]
diff = injected_image[y_slice, x_slice] - image[y_slice, x_slice]
im = ax.imshow(diff, cmap='hot', origin='lower', norm=LogNorm(vmin=0.01, vmax=max(diff.max(), 0.1)))
ax.set_title('Difference')
plt.colorbar(im, ax=ax)

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

plt.suptitle('Single Cluster Injection Test (using Actual Rubin PSF)', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Multiple Cluster Injection Test

In [ ]:
# Create injection catalog
N_CLUSTERS = 15

catalog = create_injection_catalog(
    n_clusters=N_CLUSTERS,
    image_shape=image.shape,
    mag_range=(19, 24),
    r_half_range=(5, 25),
    profile_type='plummer',
    edge_buffer=100,
    seed=42
)

print(f"Created catalog with {len(catalog)} clusters:")
print(f"{'ID':>4} {'X':>6} {'Y':>6} {'Mag':>6} {'r_half':>8}")
print("-" * 36)
for entry in catalog[:10]:
    print(f"{entry['id']:4d} {entry['x']:6.0f} {entry['y']:6.0f} {entry['magnitude']:6.1f} {entry['r_half']:8.1f}")
if len(catalog) > 10:
    print(f"... and {len(catalog)-10} more")

In [ ]:
# Inject all clusters using ACTUAL Rubin PSF
print("Injecting clusters with actual Rubin PSF...")

injected_multi, injection_info = inject_from_catalog(
    image,
    catalog,
    exposure=exposure,  # <-- Uses actual Rubin PSF!
    add_noise=False
)

print(f"\nInjection complete!")
print(f"Total flux injected: {sum(e['total_flux_injected'] for e in injection_info):.1f}")

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vmin, vmax = np.percentile(image, [1, 99])

# Original
ax = axes[0]
ax.imshow(image, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
ax.set_title('Original Coadd')

# Injected
ax = axes[1]
ax.imshow(injected_multi, cmap='gray', origin='lower', vmin=vmin, vmax=vmax)
for entry in catalog:
    color = plt.cm.plasma((entry['magnitude'] - 19) / 5)
    ax.scatter(entry['x'], entry['y'], s=100, facecolors='none', 
              edgecolors=color, linewidth=2)
ax.set_title(f'With {N_CLUSTERS} Injected Clusters')

# Difference
ax = axes[2]
diff = injected_multi - image
im = ax.imshow(diff, cmap='hot', origin='lower', norm=LogNorm(vmin=0.1, vmax=max(diff.max(), 1)))
ax.set_title('Injected Clusters Only')
plt.colorbar(im, ax=ax, label='Flux')

for ax in axes:
    ax.set_xlabel('X (pixels)')
    ax.set_ylabel('Y (pixels)')

plt.suptitle('Multiple Cluster Injection with ACTUAL Rubin PSF', fontsize=14)
plt.tight_layout()
plt.show()

## 7. PSF Spatial Variation Test

Verify that the PSF varies across the image (as expected for real coadds).

In [ ]:
# Sample PSF FWHM at many positions across the image
n_samples = 25
y_positions = np.linspace(100, image.shape[0]-100, int(np.sqrt(n_samples))).astype(int)
x_positions = np.linspace(100, image.shape[1]-100, int(np.sqrt(n_samples))).astype(int)

fwhm_map = np.zeros((len(y_positions), len(x_positions)))

for i, y in enumerate(y_positions):
    for j, x in enumerate(x_positions):
        pos = Point2D(x, y)
        fwhm = get_psf_fwhm_from_coadd(exposure, pos)
        fwhm_map[i, j] = fwhm

print(f"FWHM range: [{fwhm_map.min():.3f}, {fwhm_map.max():.3f}] pixels")
print(f"FWHM mean: {fwhm_map.mean():.3f} +/- {fwhm_map.std():.3f} pixels")

In [ ]:
# Plot PSF variation map
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
im = ax.imshow(fwhm_map, cmap='viridis', origin='lower',
               extent=[x_positions[0], x_positions[-1], y_positions[0], y_positions[-1]])
ax.set_xlabel('X (pixels)')
ax.set_ylabel('Y (pixels)')
ax.set_title('PSF FWHM Variation Across Coadd')
plt.colorbar(im, ax=ax, label='FWHM (pixels)')

ax = axes[1]
ax.hist(fwhm_map.flatten(), bins=20, edgecolor='black')
ax.axvline(fwhm_map.mean(), color='red', linestyle='--', label=f'Mean={fwhm_map.mean():.3f}')
ax.set_xlabel('PSF FWHM (pixels)')
ax.set_ylabel('Count')
ax.set_title('PSF FWHM Distribution')
ax.legend()

plt.suptitle('PSF Spatial Variation Test', fontsize=14)
plt.tight_layout()
plt.show()

## 8. Summary

In [ ]:
print("="*60)
print("INJECTION PIPELINE RSP TEST COMPLETE")
print("="*60)
print(f"\nCoadd: tract={DATA_ID['tract']}, patch={DATA_ID['patch']}, band={DATA_ID['band']}")
print(f"Image shape: {image.shape}")
print(f"PSF FWHM: {fwhm_map.mean():.3f} +/- {fwhm_map.std():.3f} pixels")
print(f"Clusters injected: {N_CLUSTERS}")
print("\nVerified:")
print("  [OK] PSF extraction from Rubin coadds")
print("  [OK] PSF normalization (sums to 1)")
print("  [OK] Convolution with actual PSF")
print("  [OK] Single cluster injection")
print("  [OK] Multiple cluster injection")
print("  [OK] PSF spatial variation")
print("\nThe pipeline is working correctly with actual Rubin PSFs!")